In [1]:
import numpy as np
import os
import sys

sys.path.insert(0, os.path.abspath('.'))
from iotools import read_AsciiGrid

# --- Set this to your catchment data folder ---
folder = ''
gis_folder = r'/Users/jpnousu/Library/CloudStorage/OneDrive-Luonnonvarakeskus/SpaFHy_RUNS/krycklan/gis/20m'

In [2]:
# All .asc files read by the model (from parameters_krycklan.py)
asc_files = {
    'catchment_mask':   'catchment_mask.asc',         # aux cmask
    'LAI_conif':        'LAI_conif.asc',               # canopy
    'LAI_decid':        'LAI_decid.asc',               # canopy
    'canopy_height':    'canopy_height.asc',           # canopy
    'canopy_fraction':  'canopy_fraction.asc',         # canopy
    'soil':             'soil.asc',                    # bucket org_id / root_id / deep_id
    'elevation':        'processed_dem.asc',           # deep soil / topmodel
    'streams':          'channels.asc',                # deep soil / aux
    'stream_distance':  'channels_distance.asc',       # deep soil
    'stream_length':    'channels_length.asc',         # deep soil
    'stream_width':     'channels_width.asc',          # deep soil
    'stream_depth':     'channels_depth.asc',          # deep soil
    'lake_mask':        'lake_mask.asc',               # deep soil / aux
    'soildepth':        'soildepth.asc',               # deep soil
    'flow_accumulation':'flow_accumulation_dinf.asc',  # topmodel
    'slope':            'slope.asc',                   # topmodel
    'twi':              'twi_dinf.asc',                # topmodel
}

In [3]:
# Read all files and collect shapes and valid-value masks
results = {}
missing = []

for name, fname in asc_files.items():
    fpath = os.path.join(gis_folder, fname)
    if not os.path.isfile(fpath):
        missing.append(name)
        continue
    data, _, _, _, _ = read_AsciiGrid(fpath, setnans=True)
    results[name] = {
        'shape': data.shape,
        'valid_mask': np.isfinite(data),   # True where data is valid (not NaN)
        'n_valid': int(np.sum(np.isfinite(data))),
    }

if missing:
    print(f'Files not found (skipped): {missing}')
print(f'Files read: {list(results.keys())}')

Files read: ['catchment_mask', 'LAI_conif', 'LAI_decid', 'canopy_height', 'canopy_fraction', 'soil', 'elevation', 'streams', 'stream_distance', 'stream_length', 'stream_width', 'stream_depth', 'lake_mask', 'soildepth', 'flow_accumulation', 'slope', 'twi']


In [4]:
# Use catchment_mask as the reference extent
ref_name = 'catchment_mask'
assert ref_name in results, f"Reference file '{ref_name}' could not be read."

ref_shape = results[ref_name]['shape']
ref_mask  = results[ref_name]['valid_mask']

print(f"Reference: {ref_name}")
print(f"  shape     : {ref_shape}")
print(f"  valid cells: {results[ref_name]['n_valid']}")
print()

all_ok = True
for name, info in results.items():
    if name == ref_name:
        continue

    shape_ok = (info['shape'] == ref_shape)
    if not shape_ok:
        print(f"[SHAPE MISMATCH] {name}: {info['shape']} vs reference {ref_shape}")
        all_ok = False
        continue

    mask_ok = np.array_equal(info['valid_mask'], ref_mask)
    if mask_ok:
        print(f"[OK]             {name}: shape {info['shape']}, valid cells {info['n_valid']}")
    else:
        diff = info['valid_mask'].astype(int) - ref_mask.astype(int)
        extra   = int(np.sum(diff ==  1))   # valid in this file but NaN in reference
        missing_cells = int(np.sum(diff == -1))  # NaN in this file but valid in reference
        print(f"[MASK MISMATCH]  {name}: valid cells {info['n_valid']} "
              f"(+{extra} extra, -{missing_cells} missing vs reference)")
        all_ok = False

print()
if all_ok:
    print("All files have the same extent of valid values.")
else:
    print("Some files have a different extent of valid values — see above.")

Reference: catchment_mask
  shape     : (572, 511)
  valid cells: 169668

[OK]             LAI_conif: shape (572, 511), valid cells 169668
[OK]             LAI_decid: shape (572, 511), valid cells 169668
[OK]             canopy_height: shape (572, 511), valid cells 169668
[OK]             canopy_fraction: shape (572, 511), valid cells 169668
[OK]             soil: shape (572, 511), valid cells 169668
[OK]             elevation: shape (572, 511), valid cells 169668
[MASK MISMATCH]  streams: valid cells 22869 (+0 extra, -146799 missing vs reference)
[MASK MISMATCH]  stream_distance: valid cells 22942 (+47 extra, -146773 missing vs reference)
[MASK MISMATCH]  stream_length: valid cells 22916 (+47 extra, -146799 missing vs reference)
[MASK MISMATCH]  stream_width: valid cells 22942 (+47 extra, -146773 missing vs reference)
[MASK MISMATCH]  stream_depth: valid cells 22895 (+0 extra, -146773 missing vs reference)
[MASK MISMATCH]  lake_mask: valid cells 1385 (+0 extra, -168283 missing vs refe

In [5]:
# --- Stream data consistency checks ---
stream_files = ['streams', 'stream_distance', 'stream_length', 'stream_width', 'stream_depth']
stream_files = [s for s in stream_files if s in results]

print("=== Stream files: cells outside catchment mask ===")
for name in stream_files:
    info = results[name]
    extra = info['valid_mask'] & ~ref_mask  # valid in stream file but outside catchment
    if np.any(extra):
        print(f"[PROBLEM] {name}: {int(np.sum(extra))} valid cell(s) outside catchment mask")
    else:
        print(f"[OK]      {name}: all valid cells are inside catchment mask")

print()
print("=== Stream files: consistent valid masks among each other ===")
# Use 'streams' as stream reference if available
stream_ref = 'streams' if 'streams' in results else stream_files[0]
stream_ref_mask = results[stream_ref]['valid_mask']
for name in stream_files:
    if name == stream_ref:
        continue
    diff = results[name]['valid_mask'].astype(int) - stream_ref_mask.astype(int)
    extra_s   = int(np.sum(diff ==  1))
    missing_s = int(np.sum(diff == -1))
    if extra_s == 0 and missing_s == 0:
        print(f"[OK]      {name}: same valid cells as '{stream_ref}'")
    else:
        print(f"[MISMATCH] {name}: +{extra_s} extra, -{missing_s} missing vs '{stream_ref}'")

print()
print("=== stream_distance: zero or negative values where streams are active ===")
if 'stream_distance' in results:
    sd_data, _, _, _, _ = read_AsciiGrid(os.path.join(gis_folder, asc_files['stream_distance']), setnans=True)
    active = results['streams']['valid_mask'] if 'streams' in results else results['stream_distance']['valid_mask']
    zero_d = active & (sd_data <= 0.0)
    if np.any(zero_d):
        print(f"[PROBLEM] stream_distance: {int(np.sum(zero_d))} cell(s) with value <= 0 where streams are active")
        rows, cols = np.where(zero_d)
        for r, c in zip(rows[:5], cols[:5]):
            print(f"  [{r},{c}] stream_distance = {sd_data[r,c]}")
    else:
        print(f"[OK]      stream_distance: all active stream cells have distance > 0")


=== Stream files: cells outside catchment mask ===
[OK]      streams: all valid cells are inside catchment mask
[PROBLEM] stream_distance: 47 valid cell(s) outside catchment mask
[PROBLEM] stream_length: 47 valid cell(s) outside catchment mask
[PROBLEM] stream_width: 47 valid cell(s) outside catchment mask
[OK]      stream_depth: all valid cells are inside catchment mask

=== Stream files: consistent valid masks among each other ===
[MISMATCH] stream_distance: +73 extra, -0 missing vs 'streams'
[MISMATCH] stream_length: +47 extra, -0 missing vs 'streams'
[MISMATCH] stream_width: +73 extra, -0 missing vs 'streams'
[MISMATCH] stream_depth: +26 extra, -0 missing vs 'streams'

=== stream_distance: zero or negative values where streams are active ===
[OK]      stream_distance: all active stream cells have distance > 0
